In [1]:
# Convert CSVs to Delta-Tables

files = [
    "olist_orders_dataset",
    "olist_order_items_dataset",
    "olist_customers_dataset",
    "olist_sellers_dataset",
    "olist_products_dataset",
    "olist_order_payments_dataset",
    "olist_geolocation_dataset",
    "product_category_name_translation"
]

for file_name in files:
    df = spark.read.option("header", "true") \
        .option("inferSchema", "true") \
        .csv(f"Files/raw/{file_name}.csv")

    
    df.write.format("delta").mode("overwrite").saveAsTable(file_name)
    print(f"{file_name}: {df.count()} records loaded")

StatementMeta(, df525597-7742-48de-a915-2f189416ded8, 3, Finished, Available, Finished, False)

olist_order_reviews_dataset: 104162 records loaded


In [4]:
# Ingest olist_order_reviews_dataset.csv into the Bronze Lakehouse as a Delta table.
#
# This file requires special CSV parsing options: review comment text can contain
# embedded line breaks and unescaped quote characters. Without handling this,
# Spark misreads row boundaries mid-review, which silently shifts values into the
# wrong columns — in this case, review_score ends up being misparsed as a date.
#
# Fix: read with multiLine=True (allows quoted fields to span multiple lines),
# explicit quote/escape characters (correctly interpret embedded quotes), and
# overwriteSchema=True (ensures the corrected schema replaces any previously
# inferred one on rewrite).

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .csv("Files/raw/olist_order_reviews_dataset.csv")
    
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_order_reviews_dataset")
print(f"reviews: {df.count()} records loaded")

StatementMeta(, f3be7888-5448-49b3-9e0a-b263f98f0516, 6, Finished, Available, Finished, False)

reviews: 99224 records
